In [4]:
library(tidyverse)
library(caret)
library(glmnet)
library(dplyr)
#library(psych) 
library(lme4)
library(e1071)
library(GGally)
options(warn=-1)
options(scipen=999)
#library(crosstable)

ERROR: Error: package or namespace load failed for ‘caret’ in dyn.load(file, DLLpath = DLLpath, ...):
 unable to load shared object '/mount/projekte9/dh/users/pageljs/R/x86_64-redhat-linux-gnu-library/4.0/stringi/libs/stringi.so':
  libicui18n.so.73: cannot open shared object file: No such file or directory


In [ ]:
seed_list<-c(1001,1111,1221,1331,1441,1551,1661,1771,1881,1991)
cut_off<-c(0)
time_off<-c(10000)
#time_off<-c(10)

lambda <- 10^seq(-3, 3, length = 100)
alpha<-seq(0,1,length=10)

In [4]:
library(doParallel)
cl <- makePSOCKcluster(20)
registerDoParallel(cl)

Loading required package: foreach


Attaching package: ‘foreach’


The following objects are masked from ‘package:purrr’:

    accumulate, when


Loading required package: iterators

Loading required package: parallel



In [5]:
corpus_list<-c("coha")

tagged_list<-c("UnTagged")

ppmi_setting_list<-c("RAW","PPMI")
comp_setting_list<-c("Aware","Agnostic")
impute_list<-c("med","na")
to_predict_list<-c("compound")#,"modifier","head")

feature_setting_list<-c("all","sim_cpf_beta","cordeiro","cosine_sim","with_setting","info_theory","freq","log_freq","family_size","prod")
caret_spearman <- function(data, lev = NULL, model = NULL) {
  spearman_val <- cor(x = data$pred, y = data$obs, method = "spearman")
  c(Spearman = spearman_val)
}

In [6]:
needed_cols<-c('modifier','head','avgModifier','stdevModifier','avgHead','stdevHead','compositionality','stdevHeadModifier','is_adj','compound','source','is_original')

cordeiro_cols<-c('arith_mean_sim.','beta.','geom_mean_sim.','sim_cpf_0.','sim_cpf_100.','sim_cpf_25.','sim_cpf_50.','sim_cpf_75.','sim_cpf_beta.')

sim_cpf_beta_cols<-c('sim_cpf_beta.')

cosine_sim_cols<-c("sim_bw_constituents.",'sim_with_head.','sim_with_modifier.')

with_setting_cols<-c("sim_bw_settings_comp.","sim_bw_settings_head.","sim_bw_settings_modifier.")
info_theory_cols<-c("local_mi.","log_ratio.","ppmi.")
log_freq_cols<-c("log_comp_freq.","log_head_freq.","log_mod_freq.")
tf_cols<-c("comp_tf.","head_tf.","mod_tf.")
freq_cols<-c("comp_freq.","head_freq.","mod_freq.")
prod_cols<-c("head_prod.","mod_prod.")
family_size_cols<-c("head_family_size.","mod_family_size.")

all_features<-c(cordeiro_cols,cosine_sim_cols,with_setting_cols,info_theory_cols,log_freq_cols,tf_cols,freq_cols,prod_cols,family_size_cols)

In [7]:
save_path<-"../Compounding/regression/"

In [10]:
m<-1
v<-1
#list_of_vi <- vector(mode="list", length=5)
#names(list_of_vi) <- c('10000', '10', '20', '50', '100')

list_of_vi_10000<-list()
list_of_vi_10<-list()
list_of_vi_20<-list()
list_of_vi_50<-list()
list_of_vi_100<-list()
list_of_rsqr=list()


for (c in corpus_list){
  for (t in tagged_list) {
    
    for (p in ppmi_setting_list){
      for (a in comp_setting_list){
        for (i in time_off) {
          
          for (j in cut_off) {
            
            for (im in impute_list) {
              print(paste0('../Compounding/datasets/',c,"/features_Compound",a,"_withSetting_",p,"_",t,"_",i,"_",j,"_",im,".csv"))
              input_df<-read.csv(paste0('../Compounding/datasets/',c,"/features_Compound",a,"_withSetting_",p,"_",t,"_",i,"_",j,"_",im,".csv"),sep = '\t')
              input_df <- input_df %>% distinct()
              input_df$is_adj<-as.logical(input_df$is_adj)
              input_df$is_original<-as.logical(input_df$is_original)
              input_df<-input_df %>% arrange(desc(comp_freq.0)) %>% filter(is_original==TRUE) %>% drop_na()
              
              train_df<-input_df %>% filter(source %in% c("reddy","cordeiro90"))
              
              test_df<-input_df %>% filter(source %in% c("cordeiro100"))
              
              df_features_train<-train_df %>% select(-needed_cols) %>% select(-one_of("comp_freq_bins"))
              df_features_train<-df_features_train %>% select(starts_with(all_features))              
              df_features_train<- Filter(function(x) sd(x) != 0,df_features_train)
              
              df_features_test<-test_df %>% select(-needed_cols) %>% select(-one_of("comp_freq_bins"))
              df_features_test<-df_features_test %>% select(starts_with(all_features))              
              df_features_test<- Filter(function(x) sd(x) != 0,df_features_test)
              
              for (f in feature_setting_list) {
                
                if(f=="cordeiro"){
                  trainX<-df_features_train %>% select(starts_with(cordeiro_cols))
                  testX<-df_features_test %>% select(starts_with(cordeiro_cols))
                }
                else if(f=="sim_cpf_beta"){
                  trainX<-df_features_train %>% select(starts_with(sim_cpf_beta_cols))
                  testX<-df_features_test %>% select(starts_with(sim_cpf_beta_cols))
                }
                else if(f=="cosine_sim"){
                  trainX<-df_features_train %>% select(starts_with(cosine_sim_cols))
                  testX<-df_features_test %>% select(starts_with(cosine_sim_cols))
                  
                }
                
                else if(f=="with_setting"){
                  trainX<-df_features_train %>% select(starts_with(with_setting_cols))
                  testX<-df_features_test %>% select(starts_with(with_setting_cols))
                }
                
                else if(f=="info_theory"){
                  trainX<-df_features_train %>% select(starts_with(info_theory_cols))
                  testX<-df_features_test %>% select(starts_with(info_theory_cols))
                }
                
                else if(f=="freq"){
                  trainX<-df_features_train %>% select(starts_with(freq_cols))
                  testX<-df_features_test %>% select(starts_with(freq_cols))
                }
                
                else if(f=="tf"){
                  trainX<-df_features_train %>% select(starts_with(tf_cols))
                  testX<-df_features_test %>% select(starts_with(tf_cols))
                }
                
                else if(f=="log_freq"){
                  trainX<-df_features_train %>% select(starts_with(log_freq_cols))
                  testX<-df_features_test %>% select(starts_with(log_freq_cols))
                }
                
                else if(f=="prod"){
                  trainX<-df_features_train %>% select(starts_with(prod_cols))
                  testX<-df_features_test %>% select(starts_with(prod_cols))
                }
                
                else if(f=="family_size"){
                  trainX<-df_features_train %>% select(starts_with(family_size_cols))
                  testX<-df_features_test %>% select(starts_with(family_size_cols))
                }
                else {
                  trainX<-df_features_train    
                  testX<-df_features_test                    
                }
                
                if (dim(trainX)[1]<10 | dim(trainX)[2]==0) {
                  print(dim(trainX))
                  break
                }
                for (pr in to_predict_list){
                  
                  if (pr=="compound") {
                    trainY<-train_df %>% select(compositionality)
                    trainY<-trainY$compositionality      
                  }
                  
                  else if (pr=="modifier") {
                    trainY<-train_df %>% select(avgModifier)
                    trainY<-trainY$avgModifier      
                  }
                  
                  else if (pr=="head") {
                    trainY<-train_df %>% select(avgHead)
                    trainY<-trainY$avgHead      
                  }                               
                  
                  print(paste0(c," ",t," ",p," ",a," ",i," ",j," ",im," ",f," ",pr))
                  
                  if (im=="na") {
                    preprocess_list<-c("nzv","medianImpute", "center", "scale")
                  }
                  else {
                    preprocess_list<-c("nzv", "center", "scale")
                  }
                  regression_error <- tryCatch( 
                    expr = {
                      elastic_model <- train(trainX,trainY,method = "glmnet",metric = "Rsquared",
                                             trControl = trainControl("repeatedcv", number = 5, repeats = 10, search="grid"),tuneGrid = expand.grid(alpha = alpha, lambda = lambda),
                                             preProcess = preprocess_list)
                      
                      
                      elastic_spearman_model <- train(trainX,trainY,method = "glmnet",metric = "Spearman",
                                                      trControl = trainControl("repeatedcv", number = 5, repeats=10,search="grid",summaryFunction = caret_spearman),tuneGrid = expand.grid(alpha = alpha, lambda = lambda),
                                                      preProcess = preprocess_list)
                    },
                    error = function(e) {e}
                  )
                  if (inherits(regression_error, "error")) {next}
                  
                  
                  
                  
                  perf_elastic<-data.frame(corpus=c,tag=t,ppmi=p,setting=a,timespan=i,cutoff=j,impute=im,features=f,y=pr,train_dim=nrow(trainX),ml_algo="elastic",method=getTrainPerf(elastic_model)[,"method"],TrainRsquared=getTrainPerf(elastic_model)[,"TrainRsquared"],TrainSpearman=getTrainPerf(elastic_spearman_model)[,"TrainSpearman"])
                  
                  varimp_elastic<-data.frame(corpus=c,tag=t,ppmi=p,setting=a,timespan=i,cutoff=j,impute=im,features=f,y=pr,train_dim=nrow(trainX),ml_algo="elastic",t(varImp(elastic_model)$importance))
                  
                  list_of_rsqr[[m]]<-perf_elastic
                  m<-m+1 
                  
                  if (i==10000) {
                    list_of_vi_10000[[v]]<-varimp_elastic
                    v<-v+1
                  }
                  else if (i==10) {
                    list_of_vi_10[[v]]<-varimp_elastic
                    v<-v+1
                  }
                  
                  else if (i==20) {
                    list_of_vi_20[[v]]<-varimp_elastic
                    v<-v+1
                  }
                  else if (i==50) {
                    list_of_vi_50[[v]]<-varimp_elastic
                    v<-v+1
                  }
                  else if (i==100) {
                    list_of_vi_100[[v]]<-varimp_elastic
                    v<-v+1
                  }                                                                                                   
                  
                }
                rsquared_df<-bind_rows(list_of_rsqr)
                rsquared_df$cutoff<-as.factor(rsquared_df$cutoff)
                write.csv(rsquared_df,paste0(save_path,"rsquared_",c,"_",t,"_",p,"_",a,"_",i,"_",j,"_",im,"_",f,"_",pr,".csv"),row.names = FALSE)
                list_of_rsqr<-list()
                if (i==10000) {
                  varimp_10000_df<-bind_rows(list_of_vi_10000)
                  varimp_10000_df$cutoff<-as.factor(varimp_10000_df$cutoff)
                  varimp_10000_df[is.na(varimp_10000_df)] <- 0
                  write.csv(varimp_10000_df,paste0(save_path,"varimp_10000_",c,"_",t,"_",p,"_",a,"_",i,"_",j,"_",im,"_",f,"_",pr,".csv"),row.names = FALSE)
                  list_of_vi_10000<-list()
                  
                }
                else if (i==10) {
                  varimp_10_df<-bind_rows(list_of_vi_10)
                  varimp_10_df$cutoff<-as.factor(varimp_10_df$cutoff)
                  varimp_10_df[is.na(varimp_10_df)] <- 0
                  write.csv(varimp_10_df,paste0(save_path,"varimp_10_",c,"_",t,"_",p,"_",a,"_",i,"_",j,"_",im,"_",f,"_",pr,".csv"),row.names = FALSE)
                  list_of_vi_10<-list()
                }
                
                else if (i==20) {
                  varimp_20_df<-bind_rows(list_of_vi_20)
                  varimp_20_df$cutoff<-as.factor(varimp_20_df$cutoff)
                  varimp_20_df[is.na(varimp_20_df)] <- 0
                  write.csv(varimp_20_df,paste0(save_path,"varimp_20_",c,"_",t,"_",p,"_",a,"_",i,"_",j,"_",im,"_",f,"_",pr,".csv"),row.names = FALSE)
                  list_of_vi_20<-list()
                  
                }
                else if (i==50) {
                  varimp_50_df<-bind_rows(list_of_vi_50)
                  varimp_50_df$cutoff<-as.factor(varimp_50_df$cutoff)
                  varimp_50_df[is.na(varimp_50_df)] <- 0
                  write.csv(varimp_50_df,paste0(save_path,"varimp_50_",c,"_",t,"_",p,"_",a,"_",i,"_",j,"_",im,"_",f,"_",pr,".csv"),row.names = FALSE)
                  list_of_vi_50<-list()
                  
                }
                else if (i==100) {
                  varimp_100_df<-bind_rows(list_of_vi_100)
                  varimp_100_df$cutoff<-as.factor(varimp_100_df$cutoff)
                  varimp_100_df[is.na(varimp_100_df)] <- 0
                  write.csv(varimp_100_df,paste0(save_path,"varimp_100_",c,"_",t,"_",p,"_",a,"_",i,"_",j,"_",im,"_",f,"_",pr,".csv"),row.names = FALSE)
                  list_of_vi_100<-list()
                  
                } 
                
                
              }
            }
          }
        }
      }
    }
  }
  
  
}

[1] "../Compounding/datasets/coha/features_CompoundAware_withSetting_RAW_UnTagged_10000_0_med.csv"
[1] "coha UnTagged RAW Aware 10000 0 med all compound"
[1] "coha UnTagged RAW Aware 10000 0 med sim_cpf_beta compound"
Something is wrong; all the Rsquared metric values are missing:
      RMSE         Rsquared         MAE      
 Min.   : NA    Min.   : NA    Min.   : NA   
 1st Qu.: NA    1st Qu.: NA    1st Qu.: NA   
 Median : NA    Median : NA    Median : NA   
 Mean   :NaN    Mean   :NaN    Mean   :NaN   
 3rd Qu.: NA    3rd Qu.: NA    3rd Qu.: NA   
 Max.   : NA    Max.   : NA    Max.   : NA   
 NA's   :1000   NA's   :1000   NA's   :1000  
[1] "coha UnTagged RAW Aware 10000 0 med cordeiro compound"
[1] "coha UnTagged RAW Aware 10000 0 med cosine_sim compound"
[1] "coha UnTagged RAW Aware 10000 0 med with_setting compound"
[1] "coha UnTagged RAW Aware 10000 0 med info_theory compound"
[1] "coha UnTagged RAW Aware 10000 0 med freq compound"
[1] "coha UnTagged RAW Aware 10000 0 med log_

In [11]:
rsquared_df

corpus,tag,ppmi,setting,timespan,cutoff,impute,features,y,train_dim,ml_algo,method,TrainRsquared,TrainSpearman
<chr>,<chr>,<chr>,<chr>,<dbl>,<fct>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>
coha,UnTagged,PPMI,Agnostic,10000,0,na,cosine_sim,compound,167,elastic,glmnet,0.1523281,0.4026757


In [172]:
train(trainX,trainY,method = "glmnet",metric = "Rsquared",
    trControl = trainControl("repeatedcv", number = 5, repeats = 10, 
    search="grid"),tuneGrid = expand.grid(alpha = alpha, lambda = lambda),
                                         preProcess = preprocess_list)

ERROR: Error in unserialize(node$con): unknown input format


In [165]:
rsquared_df

corpus,tag,ppmi,setting,timespan,cutoff,impute,features,y,train_dim,ml_algo,method,TrainRsquared,TrainSpearman
<chr>,<chr>,<chr>,<chr>,<dbl>,<fct>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>
coha,UnTagged,RAW,Aware,10000,0,med,prod,compound,180,elastic,glmnet,0.06926665,0.2483605


In [161]:
dim(trainY)

NULL

In [153]:
df_features_X %>% select(starts_with(with_setting_cols))

1  
2  
3  
4  
5  
6  
7  
8  
9  
10 
11 
12 
13 
14 
15 
16 
17 
18 
19 
20 
21 
22 
23 
24 
25 
26 
27 
28 
29 
30 
...
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180

In [155]:
colnames(df_features_X)

[1] "arith_mean_sim.0"      "beta.0"                "geom_mean_sim.0"      
 [4] "sim_cpf_0.0"           "sim_cpf_100.0"         "sim_cpf_25.0"         
 [7] "sim_cpf_50.0"          "sim_cpf_75.0"          "sim_cpf_beta.0"       
[10] "sim_bw_constituents.0" "sim_with_head.0"       "sim_with_modifier.0"  
[13] "local_mi.0"            "log_ratio.0"           "ppmi.0"               
[16] "log_comp_freq.0"       "log_head_freq.0"       "log_mod_freq.0"       
[19] "comp_tf.0"             "head_tf.0"             "mod_tf.0"             
[22] "comp_freq.0"           "head_freq.0"           "mod_freq.0"           
[25] "head_prod.0"           "mod_prod.0"            "head_family_size.0"   
[28] "mod_family_size.0"